# Silver Layer - Enriched Health Datasets

This notebook reads from `health_catalog.bronze.*`, enriches and normalizes the data, and writes to `health_catalog.silver.*`.

In [ ]:
from pyspark.sql import SparkSession
from databricks_health import SilverEnrichmentOrchestrator

spark = SparkSession.builder.getOrCreate()

def main():
    orchestrator = SilverEnrichmentOrchestrator(spark, catalog='health_catalog')
    orchestrator.run()

if __name__ == '__main__':
    main()


## Silver SQL Views and Queries

### Infrastructure health
````sql
CREATE OR REPLACE VIEW health_catalog.silver.infra_health_view AS
SELECT
  cluster_id,
  cluster_name,
  state,
  monitor_startup_attempts,
  monitor_startup_success,
  monitor_startup_success / NULLIF(monitor_startup_attempts, 0) AS monitor_startup_success_rate,
  cluster_age_hours
FROM health_catalog.silver.infra_health
````

### Pipeline health
````sql
CREATE OR REPLACE VIEW health_catalog.silver.pipeline_health_view AS
SELECT
  job_id,
  name AS job_name,
  state AS pipeline_state,
  result_state,
  count(*) OVER (PARTITION BY job_id) AS total_runs,
  sum(is_failed) OVER (PARTITION BY job_id) AS failed_runs,
  avg(duration_minutes) OVER (PARTITION BY job_id) AS avg_duration_minutes
FROM health_catalog.silver.pipeline_health
````

### Data product health
````sql
CREATE OR REPLACE VIEW health_catalog.silver.data_product_health_view AS
SELECT
  catalog_name,
  schema_name,
  table_name,
  row_count,
  last_access_time,
  CASE WHEN last_access_time < current_timestamp() - INTERVAL 7 DAYS THEN 'stale' ELSE 'active' END AS access_status
FROM health_catalog.silver.table_accessibility
````